In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib, numpy as np, pandas as pd, torch
CODE = "/content/drive/MyDrive/esci/code"; sys.path.insert(0, CODE)
import metrics, cross_encoder as C, train_cross_encoder as T
for m in (metrics, C, T): importlib.reload(m)
from metrics import evaluate

BASE = "/content/drive/MyDrive/esci"
df = pd.read_parquet(f"{BASE}/processed/rerank_pairs_v2.3.parquet")
qs = pd.read_parquet(f"{BASE}/dataset_v2.3/esci_electronics_v2.3_queries.parquet")[["query_id", "rule", "scope_group"]]
df = df.merge(qs, on="query_id", how="left")
assert df["rule"].notna().all()

# leak checks (test only used here, then dropped)
ids = {p: set(df.loc[df.part == p, "query_id"]) for p in ["train", "val", "test"]}
assert not (ids["train"] & ids["val"]) and not (ids["train"] & ids["test"]) and not (ids["val"] & ids["test"])
norm = lambda s: " ".join(str(s).lower().split())
assert not (set(df.loc[df.part == "train", "query"].map(norm)) & set(df.loc[df.part == "val", "query"].map(norm)))
train = df[df.part == "train"].reset_index(drop=True)
val   = df[df.part == "val"].reset_index(drop=True)
del df

# M3 category stand-in: Rule A only (Rule B category comes from labelled products = leak)
CAT = {"mobile_computing": "Mobile & Computing", "components_accessories": "Components & Accessories",
       "home_electronics": "Home Electronics", "imaging": "Cameras & Imaging", "audio": "Audio",
       "computer_peripherals": "Computer Peripherals", "gaming": "Gaming", "smart_home": "Smart Home"}
for d in (train, val):
    d["category"] = np.where(d["rule"] == "A", d["scope_group"].map(CAT), None)
    d["q_raw"]         = d["query"].map(C._clean)
    d["q_raw_cat"]     = [C.query_to_text(q, category=c, use_rules=False) for q, c in zip(d["query"], d["category"])]
    d["q_focused"]     = [C.query_to_text(q) for q in d["query"]]
    d["q_focused_cat"] = [C.query_to_text(q, category=c) for q, c in zip(d["query"], d["category"])]
    prods = [C.esci_row_to_product(r) for r in d.to_dict("records")]
    for s in C.STYLES:
        d[f"text_{s}"] = [C.product_to_text(p, s) for p in prods]

QUERY_INPUTS = ["q_raw", "q_raw_cat", "q_focused", "q_focused_cat"]
print("product formats:", C.STYLES, "| same in trainer:", T.STYLES == C.STYLES)
print("train rows", len(train), "| val rows", len(val), "| GPU", torch.cuda.get_device_name(0))
print("val queries with category:", val.loc[val.category.notna(), "query_id"].nunique(), "of", val.query_id.nunique())
ex = val.drop_duplicates("query_id").sample(4, random_state=3)
print(ex[["query"] + QUERY_INPUTS].to_string())
print(val[["text_title", "text_key_value"]].head(2).to_string())

Mounted at /content/drive
product formats: ['title', 'title_brand', 'full', 'full_bullets', 'key_value'] | same in trainer: True
train rows 65852 | val rows 11741 | GPU Tesla T4
val queries with category: 495 of 656
                         query                     q_raw                                   q_raw_cat                 q_focused                               q_focused_cat
4488     galaxy watch active 2     galaxy watch active 2  [Mobile & Computing] galaxy watch active 2     galaxy watch active 2  [Mobile & Computing] galaxy watch active 2
736                a8g oled tv               a8g oled tv              [Home Electronics] a8g oled tv               a8g oled tv              [Home Electronics] a8g oled tv
1892                  aux cord                  aux cord                                    aux cord                  aux cord                                    aux cord
8364  overhead headphone wired  overhead headphone wired            [Audio] overhead headphone wired

In [2]:
MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LOG_FO = "s9_fullonly_val_log.csv"            # separate log, so the main grid log stays clean
for seed in (42, 43, 44):
    cfg = {**T.CFG, "query_col": "q_focused_cat", "styles": ["full"], "eval_styles": C.STYLES, "seed": seed}
    name = "minilm_l6_fullonly__q_focused_cat" + ("" if seed == 42 else f"__s{seed}")
    print(f"\n===== {name} =====")
    T.train_run(MODEL, name, train, val, evaluate, BASE, cfg=cfg, log_name=LOG_FO)


===== minilm_l6_fullonly__q_focused_cat =====
minilm_l6_fullonly__q_focused_cat: starting at epoch 1 from cross-encoder/ms-marco-MiniLM-L-6-v2


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

...

epoch 1 done: loss 0.4783, 1.3 min
       style  ndcg@10    mrr    p@5
       title   0.9225 0.9385 0.8473
 title_brand   0.9244 0.9405 0.8470
        full   0.9238 0.9398 0.8460
full_bullets   0.9279 0.9410 0.8451
   key_value   0.9256 0.9394 0.8470
  epoch 2 step 500/2058 loss 0.3426
  epoch 2 step 1000/2058 loss 0.3376
  epoch 2 step 1500/2058 loss 0.3363
  epoch 2 step 2000/2058 loss 0.3349


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 2 done: loss 0.3341, 1.3 min
       style  ndcg@10    mrr    p@5
       title   0.9271 0.9446 0.8497
 title_brand   0.9285 0.9458 0.8494
        full   0.9302 0.9448 0.8494
full_bullets   0.9309 0.9508 0.8473
   key_value   0.9295 0.9423 0.8500
  epoch 3 step 500/2058 loss 0.3065
  epoch 3 step 1000/2058 loss 0.3056
  epoch 3 step 1500/2058 loss 0.3033
  epoch 3 step 2000/2058 loss 0.3019


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 3 done: loss 0.3017, 1.3 min
       style  ndcg@10    mrr    p@5
       title   0.9271 0.9469 0.8488
 title_brand   0.9273 0.9439 0.8479
        full   0.9277 0.9439 0.8470
full_bullets   0.9283 0.9462 0.8470
   key_value   0.9271 0.9419 0.8485


In [2]:
LOG_FO = "s9_fullonly_val_log.csv"

In [3]:
from scipy.stats import wilcoxon

def load_best(log_file, prefix):
    L = pd.read_csv(f"{BASE}/results/{log_file}")
    L = L[L.run.str.startswith(prefix + "__q_focused_cat")].copy()
    L["seed"] = L.run.str.extract(r"__s(\d+)$")[0].fillna("42").astype(int)
    ep = L.groupby(["run", "seed", "epoch"])["ndcg@10"].mean().reset_index()          # same epoch rule as before
    best = ep.loc[ep.groupby("run")["ndcg@10"].idxmax()][["run", "seed", "epoch"]]
    return L.merge(best, on=["run", "seed", "epoch"]), best

mix, mix_best = load_best("s7_grid_val_log.csv", "minilm_l6")             # your chosen model (5-format mix)
fo,  fo_best  = load_best(LOG_FO,               "minilm_l6_fullonly")     # trained on 'full' only
print("best epochs  mix:", mix_best[["seed", "epoch"]].values.tolist(), "| full-only:", fo_best[["seed", "epoch"]].values.tolist())

# ---- Part 1 + 2: per-format table, mean ± std over 3 seeds
def fmt_table(d, label):
    t = d.groupby("style")[["ndcg@10", "mrr", "p@5"]].agg(["mean", "std"])
    t.columns = [f"{label}_{m}_{s}" for m, s in t.columns]; return t
table = fmt_table(mix, "mix").join(fmt_table(fo, "fullonly")).loc[C.STYLES]
show = table[["mix_ndcg@10_mean", "mix_ndcg@10_std", "fullonly_ndcg@10_mean", "fullonly_ndcg@10_std"]].copy()
show["diff (mix - fullonly)"] = show["mix_ndcg@10_mean"] - show["fullonly_ndcg@10_mean"]
print("\nNDCG@10 per product format (validation, 3 seeds):\n", show.round(4).to_string())

# ---- per format: paired Wilcoxon (per-query NDCG, averaged over the 3 seeds)
def perq(best_df, style):
    frames = []
    for _, r in best_df.iterrows():
        d = pd.read_csv(f"{BASE}/models/{r.run}/per_query_val_epoch_{int(r.epoch)}.csv")
        frames.append(d[d["style"] == style].set_index("query_id")["ndcg@10"])
    return pd.concat(frames, axis=1).mean(axis=1)
print()
for s in C.STYLES:
    a, b = perq(mix_best, s), perq(fo_best, s)
    d = a - b.loc[a.index]; nz = d[d != 0]
    print(f"{s:13s}: mix - fullonly = {d.mean():+.4f} | better {int((d>0).sum())} / worse {int((d<0).sum())} | Wilcoxon p = {wilcoxon(nz).pvalue:.4f}")

# ---- spread across formats = how much the score depends on which fields a platform gives
for lbl in ("mix", "fullonly"):
    m = table[f"{lbl}_ndcg@10_mean"]
    print(f"{lbl:9s}: best format {m.max():.4f}, worst {m.min():.4f}, gap {m.max()-m.min():.4f}")
table.to_csv(f"{BASE}/results/s9_format_table.csv")

best epochs  mix: [[42, 3], [43, 3], [44, 2]] | full-only: [[42, 2], [43, 3], [44, 2]]

NDCG@10 per product format (validation, 3 seeds):
               mix_ndcg@10_mean  mix_ndcg@10_std  fullonly_ndcg@10_mean  fullonly_ndcg@10_std  diff (mix - fullonly)
style                                                                                                              
title                   0.9270           0.0011                 0.9270                0.0005                -0.0001
title_brand             0.9281           0.0009                 0.9276                0.0008                 0.0005
full                    0.9278           0.0012                 0.9287                0.0014                -0.0008
full_bullets            0.9286           0.0012                 0.9290                0.0016                -0.0005
key_value               0.9280           0.0011                 0.9284                0.0010                -0.0004

title        : mix - fullonly = -0.0000 | better

In [2]:
# ===== SETUP (run first after every restart) =====
from google.colab import drive
drive.mount('/content/drive')
import sys, importlib, re, numpy as np, pandas as pd, torch
from scipy.stats import wilcoxon
CODE = "/content/drive/MyDrive/esci/code"; sys.path.insert(0, CODE)
import metrics, cross_encoder as C, train_cross_encoder as T
for m in (metrics, C, T): importlib.reload(m)
from metrics import evaluate
BASE = "/content/drive/MyDrive/esci"
LOG_FO = "s9_fullonly_val_log.csv"

# validation data only (scoring needs no training data)
df = pd.read_parquet(f"{BASE}/processed/rerank_pairs_v2.3.parquet")
qs = pd.read_parquet(f"{BASE}/dataset_v2.3/esci_electronics_v2.3_queries.parquet")[["query_id", "rule", "scope_group"]]
val = df[df.part == "val"].merge(qs, on="query_id", how="left").reset_index(drop=True); del df
CAT = {"mobile_computing": "Mobile & Computing", "components_accessories": "Components & Accessories",
       "home_electronics": "Home Electronics", "imaging": "Cameras & Imaging", "audio": "Audio",
       "computer_peripherals": "Computer Peripherals", "gaming": "Gaming", "smart_home": "Smart Home"}
val["category"] = np.where(val["rule"] == "A", val["scope_group"].map(CAT), None)
val["q_focused_cat"] = [C.query_to_text(q, category=c) for q, c in zip(val["query"], val["category"])]
prods = [C.esci_row_to_product(r) for r in val.to_dict("records")]
for s in C.STYLES:
    val[f"text_{s}"] = [C.product_to_text(p, s) for p in prods]

# best epoch per run (same rule as before)
def load_best(log_file, prefix):
    L = pd.read_csv(f"{BASE}/results/{log_file}")
    L = L[L.run.str.startswith(prefix + "__q_focused_cat")].copy()
    L["seed"] = L.run.str.extract(r"__s(\d+)$")[0].fillna("42").astype(int)
    ep = L.groupby(["run", "seed", "epoch"])["ndcg@10"].mean().reset_index()
    best = ep.loc[ep.groupby("run")["ndcg@10"].idxmax()][["run", "seed", "epoch"]]
    return L.merge(best, on=["run", "seed", "epoch"]), best
mix, mix_best = load_best("s7_grid_val_log.csv", "minilm_l6")
fo,  fo_best  = load_best(LOG_FO, "minilm_l6_fullonly")

print("val rows:", len(val), "| GPU:", torch.cuda.is_available() and torch.cuda.get_device_name(0))
print("mix best epochs:", mix_best[["seed", "epoch"]].values.tolist())
print("full-only best epochs:", fo_best[["seed", "epoch"]].values.tolist())

Mounted at /content/drive
val rows: 11741 | GPU: Tesla T4
mix best epochs: [[42, 3], [43, 3], [44, 2]]
full-only best epochs: [[42, 2], [43, 3], [44, 2]]


In [3]:
import re
p = val.drop_duplicates("product_id")
def inside(title, part):
    t, x = C._clean(title).lower(), C._clean(part).lower()
    return (x in t) if x else np.nan
brand_in = [inside(t, b) for t, b in zip(p.product_title, p.product_brand)]
color_in = [inside(t, c) for t, c in zip(p.product_title, p.product_color)]
print("val products:", len(p))
print(f"brand written inside the title: {np.nanmean(brand_in):.1%}  (of products that have a brand)")
print(f"color written inside the title: {np.nanmean(color_in):.1%}  (of products that have a color)")
print(f"products with bullet points: {p.product_bullet_point.notna().mean():.1%}")
print(f"median title length: {int(p.product_title.str.split().str.len().median())} words")

val products: 9853
brand written inside the title: 77.5%  (of products that have a brand)
color written inside the title: 49.3%  (of products that have a color)
products with bullet points: 91.8%
median title length: 22 words


In [4]:
def first_words(t, n=4): return " ".join(C._clean(t).split()[:n])
def drop_brand(t, b):
    t, b = C._clean(t), C._clean(b)
    out = " ".join(re.sub(re.escape(b), " ", t, flags=re.I).split()) if b else t
    return out or t
def kv(r, with_title=False, title_last=True, bullets=False):
    parts = [f"brand: {C._clean(r['product_brand'])}" if C._clean(r['product_brand']) else "",
             f"color: {C._clean(r['product_color'])}" if C._clean(r['product_color']) else ""]
    if bullets and C._clean(r['product_bullet_point']):
        parts.append("features: " + C._clean(r['product_bullet_point'])[:200])
    if with_title: parts = parts + [C._clean(r['product_title'])] if title_last else [C._clean(r['product_title'])] + parts
    return " | ".join(x for x in parts if x)

HARD = {
    "title_4words":   lambda r: first_words(r["product_title"]),                  # very short listing title
    "title_no_brand": lambda r: drop_brand(r["product_title"], r["product_brand"]),
    "title_noisy":    lambda r: " ".join(re.sub(r"[^a-z0-9 ]", " ", C._clean(r["product_title"]).lower()).split()),
    "kv_title_last":  lambda r: kv(r, with_title=True, title_last=True),           # fields in a different order
    "no_title":       lambda r: kv(r, bullets=True),                                # specs only, no title
}
recs = val.to_dict("records")
for name, f in HARD.items():
    val[f"text_{name}"] = [f(r) for r in recs]
print(val[[f"text_{n}" for n in HARD]].head(3).to_string())
print("\nempty 'no_title' texts:", int((val["text_no_title"] == "").sum()), "of", len(val))

                    text_title_4words                                                                                                                                                        text_title_no_brand                                                                                                                                                          text_title_noisy                                                                                                                                                                                                            text_kv_title_last                                                                                                                                                                                                                                                text_no_title
0  PANASONIC Amplified Cordless Phone                                                                                  Amplified Cordless Phone wi

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
ALL_FORMATS = C.STYLES + list(HARD)
CKPTS = {"zero_shot": [("zero_shot", "cross-encoder/ms-marco-MiniLM-L-6-v2")]}
CKPTS["fullonly"] = [(r.run, f"{BASE}/models/{r.run}/epoch_{int(r.epoch)}") for _, r in fo_best.iterrows()]
CKPTS["mix"]      = [(r.run, f"{BASE}/models/{r.run}/epoch_{int(r.epoch)}") for _, r in mix_best.iterrows()]

rows, PQ = [], {}
qs = val["q_focused_cat"].tolist()
for group, items in CKPTS.items():
    for run, path in items:
        tok = AutoTokenizer.from_pretrained(path)
        m = AutoModelForSequenceClassification.from_pretrained(path).to("cuda").eval()
        for f in ALL_FORMATS:
            v = val[["query_id", "gain"]].copy()
            v["score"] = T.score_pairs(m, tok, qs, val[f"text_{f}"].tolist(), 256, 128)
            mtr, pq = evaluate(v)
            rows.append({"group": group, "run": run, "format": f, **mtr})
            PQ[(group, run, f)] = pq.set_index("query_id")["ndcg@10"]
        del m; torch.cuda.empty_cache(); print("done", run)

R = pd.DataFrame(rows)
summary = R.groupby(["format", "group"])["ndcg@10"].agg(["mean", "std"]).unstack("group")
summary = summary.reindex(ALL_FORMATS)
summary[("diff", "mix-fullonly")] = summary[("mean", "mix")] - summary[("mean", "fullonly")]
print(summary.round(4).to_string())

print()
for f in ALL_FORMATS:
    a = pd.concat([PQ[("mix", r, f)] for r, _ in CKPTS["mix"]], axis=1).mean(axis=1)
    b = pd.concat([PQ[("fullonly", r, f)] for r, _ in CKPTS["fullonly"]], axis=1).mean(axis=1)
    d = a - b.loc[a.index]; nz = d[d != 0]
    print(f"{f:15s}: mix - fullonly = {d.mean():+.4f} | Wilcoxon p = {wilcoxon(nz).pvalue:.4f}")
R.to_csv(f"{BASE}/results/s9_hard_formats.csv", index=False)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

...

done minilm_l6__q_focused_cat


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

done minilm_l6__q_focused_cat__s43


Loading weights:   0%|          | 0/105 [00:01<?, ?it/s]

done minilm_l6__q_focused_cat__s44
                   mean                        std                           diff
group          fullonly     mix zero_shot fullonly     mix zero_shot mix-fullonly
format                                                                           
title            0.9270  0.9270    0.9090   0.0005  0.0011       NaN      -0.0001
title_brand      0.9276  0.9281    0.9116   0.0008  0.0009       NaN       0.0005
full             0.9287  0.9278    0.9119   0.0014  0.0012       NaN      -0.0008
full_bullets     0.9290  0.9286    0.9090   0.0016  0.0012       NaN      -0.0005
key_value        0.9284  0.9280    0.9122   0.0010  0.0011       NaN      -0.0004
title_4words     0.8815  0.8811    0.8760   0.0016  0.0007       NaN      -0.0004
title_no_brand   0.8995  0.9008    0.8812   0.0011  0.0018       NaN       0.0013
title_noisy      0.9246  0.9239    0.9078   0.0006  0.0010       NaN      -0.0007
kv_title_last    0.9269  0.9270    0.9099   0.0023  0.0025     